## STEP 6 - Matcher: resume → job scores (+ smoke test)

In [1]:
# 6) Matcher — load artifacts and score
import re
import pandas as pd
from pathlib import Path
from scipy.sparse import load_npz, csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
%run ./hidden.ipynb  # sets datasets_folder_path

P = datasets_folder_path / "final" / "jobs_skills" / "vocab"
df_jobs = pd.read_parquet(P / "jobs_clean.parquet")
X_jobs  = load_npz(P / "jobs_vectors.npz")
vocab   = pd.read_csv(P / "skills_vocab.csv", header=None)[0].tolist()
skill_index = {s:i for i,s in enumerate(vocab)}

# Keep a small stoplist (optional), filter overly-generic skills (do later if needed)
STOP_SKILLS = set([])  # e.g., {"communication","leadership","operations"}

def extract_resume_skills(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    # allow: a–z, 0–9, + . # / - and spaces
    clean = re.sub(r"[^a-z0-9+.#/\- ]", " ", text.lower())  # note the hyphen is at the end of the class
    clean = " " + re.sub(r"\s+", " ", clean).strip() + " "
    hits = [s for s in vocab if f" {s} " in clean and s not in STOP_SKILLS]
    return sorted(set(hits))


def vec_from_skills(skills: list[str], n: int):
    idx = [skill_index[s] for s in skills if s in skill_index]
    if not idx:
        return csr_matrix((1, n), dtype=int)
    return csr_matrix(([1]*len(idx), ([0]*len(idx), idx)), shape=(1, n), dtype=int)

def match_jobs(resume_text: str, field: str|None=None, topn: int=20):
    have = extract_resume_skills(resume_text)
    x = vec_from_skills(have, len(vocab))
    scores = cosine_similarity(x, X_jobs).ravel()
    out = df_jobs.copy()
    out["score"] = scores
    if field and field != "Any":
        out = out[out["field"].str.lower() == field.lower()]
    have_set = set(have)
    out["missing_skills"] = out["skills_final"].apply(lambda lst: sorted(set(lst) - have_set)[:8])
    cols = ["score","job_title","field","skills_final","missing_skills","job_description","qualifications"]
    return out.sort_values("score", ascending=False).head(topn)[cols], have

# Smoke test
demo = "Python, SQL, Tableau, Docker, AWS; built ETL pipelines and dashboards."
top, have = match_jobs(demo, field="Data", topn=10)
print("Detected resume skills:", have)

# Display top matches
print(top[["score","job_title","field","missing_skills"]].head(5).to_string(index=False))


Python executable: /media/artir/Data1/coding/projects/Final/GitHub/AI-ML_Projects/.venv/bin/python
Enable user-site: False
Datasets folder loaded from project root
Detected resume skills: ['aws', 'docker', 'etl', 'python', 'sql', 'tableau']
   score               job_title field                                        missing_skills
0.680414            Data Analyst  Data             [airflow, hadoop, informatica, snowflake]
0.615457 Analytics ETL Developer  Data [azure, collaboration, javascript, power bi, r, ssis]
0.612372           Data Engineer  Data                                                [hive]
0.612372     senior data analyst  Data                                          [operations]
0.612372           Data Engineer  Data                                                [hive]


## STEP 7 - Gap Analysis & Project Recommendations

It:
- calls your matcher to get the top jobs,
- recomputes per-job missing skills robustly,
- ranks gaps by (score-weighted) frequency across the top matches,
- loads project_ideas.json (creates a sensible default if missing), and
- returns a tidy table of 3–5 project suggestions mapped to the most impactful gaps, plus a compact “gap profile” you can show in the UI.

Assumption: You already have match_jobs(resume_text, field, topn) and extract_resume_skills(resume_text) from Step 6 (you do—your smoke test used them). If your top DataFrame already includes missing_skills, the code will still work (it recomputes to be safe).

In [2]:
# step7_gap_to_projects.py

from __future__ import annotations
import json, re, math, random
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple
import pandas as pd

# ---------- Config ----------
DATA_DIR = datasets_folder_path / "preared"
PROJECT_IDEAS_PATH = Path("project_ideas.json")  # will be created if missing
TOPN_JOBS_FOR_GAPS = 30
N_PROJECTS = 5
SEED = 7
random.seed(SEED)

# ---------- Utilities ----------
def _norm_skill(s: str) -> str:
    """Normalize skill tokens to a canonical, lowercase form."""
    s = s.strip().lower()
    # quick normalization for common variants/hyphens
    s = s.replace("scikit learn", "scikit-learn").replace("powerbi", "power bi")
    s = s.replace("git/github", "git").replace("ms excel","excel")
    s = re.sub(r"\s+", " ", s)
    return s

def _choose_skill_col(df: pd.DataFrame) -> str:
    """Find the column that holds job skills."""
    for c in ["skills", "skill_list", "job_skills", "skills_parsed", "skills_set"]:
        if c in df.columns:
            return c
    # last resort: try to infer
    for c in df.columns:
        if df[c].apply(lambda x: isinstance(x, (list, set, tuple))).mean() > 0.7:
            return c
    raise KeyError("Could not find a skills column in the jobs DataFrame.")

def _ensure_list(x):
    if x is None:
        return []
    if isinstance(x, (list, set, tuple)):
        return list(x)
    return [x]

def _estimate_project_count(resume_text: str) -> int:
    """
    Simple heuristic: count lines/segments that look like projects.
    If you already have a resume parser, replace this with it.
    """
    text = resume_text.lower()
    hits = re.findall(r"\bproject[s]?:|\bcapstone\b|\bhackathon\b|\bcase study\b", text)
    bulletish = len(re.findall(r"[\n•\-–]\s*[A-Z0-9]", resume_text))
    return max(len(hits), min(bulletish // 6, 6))  # very rough, safe default

# ---------- Project ideas store ----------
_DEFAULT_IDEAS: Dict[str, List[Dict[str, Any]]] = {
    # Data / Analytics
    "sql": [
        {"title": "SQL Data Mart for Sales KPIs",
         "summary": "Design a star schema + SQL transforms to serve daily KPIs.",
         "stack": ["SQL", "dbt (optional)"],
         "deliverables": ["DDL/DML scripts", "ERD", "5+ KPI queries"]},
    ],
    "python": [
        {"title": "Data Quality Toolkit",
         "summary": "Build a Python CLI to run validation checks on CSV/Parquet.",
         "stack": ["Python", "pandas", "pytest"],
         "deliverables": ["CLI with 10+ checks", "README", "unit tests"]},
    ],
    "pandas": [
        {"title": "Retail Cohort & RFM Analysis",
         "summary": "Notebook that computes cohorts, RFM, and retention plots.",
         "stack": ["Python", "pandas", "matplotlib"],
         "deliverables": ["Notebook", "plots", "insights.md"]},
    ],
    "tableau": [
        {"title": "Executive KPI Dashboard",
         "summary": "Interactive dashboard with drill-downs and filters.",
         "stack": ["Tableau"],
         "deliverables": [".twb(x)", "dashboard screenshots"]},
    ],
    "power bi": [
        {"title": "Power BI Supply Chain Dashboard",
         "summary": "End-to-end model + visuals for inventory/lead-time.",
         "stack": ["Power BI"],
         "deliverables": [".pbix", "DAX measures", "README"]},
    ],
    "excel": [
        {"title": "Excel Financial Model",
         "summary": "3-statement model with scenario toggles and charts.",
         "stack": ["Excel"],
         "deliverables": ["xlsx model", "assumptions sheet"]},
    ],
    "scikit-learn": [
        {"title": "Churn Prediction Pipeline",
         "summary": "Supervised baseline with proper CV & feature importances.",
         "stack": ["Python", "scikit-learn"],
         "deliverables": ["notebook", "ROC/PR curves", "report.md"]},
    ],
    "nlp": [
        {"title": "Ticket Auto-Tagging",
         "summary": "Vectorize text and train a multi-label tagger for support tickets.",
         "stack": ["Python", "scikit-learn", "spaCy or HuggingFace"],
         "deliverables": ["notebook", "evaluation", "inference script"]},
    ],
    "spark": [
        {"title": "Clickstream ETL on Spark",
         "summary": "Batch ETL + sessionization + aggregations over 100M rows.",
         "stack": ["PySpark"],
         "deliverables": ["Spark job", "README", "sample data"]},
    ],
    "airflow": [
        {"title": "Airflow ETL DAGs",
         "summary": "Daily ingestion + transform + QA with SLAs and retries.",
         "stack": ["Airflow", "Python"],
         "deliverables": ["DAGs", "operators", "README"]},
    ],
    "docker": [
        {"title": "Containerized Analytics Stack",
         "summary": "Dockerize ETL + notebook + DB; one-command dev env.",
         "stack": ["Docker", "docker-compose"],
         "deliverables": ["Dockerfiles", "compose.yml"]},
    ],
    "aws": [
        {"title": "AWS Data Lake Mini",
         "summary": "S3 + Glue + Athena + QuickSight demo pipeline.",
         "stack": ["AWS S3", "Glue", "Athena"],
         "deliverables": ["Infra diagram", "SQL queries", "README"]},
    ],
    "gcp": [
        {"title": "BigQuery ELT + Looker Studio",
         "summary": "Load → transform in BQ → visualize in Looker Studio.",
         "stack": ["GCP BigQuery"],
         "deliverables": ["SQL", "dashboard link", "README"]},
    ],
    "fastapi": [
        {"title": "ML Inference API",
         "summary": "Serve a model behind FastAPI with input validation.",
         "stack": ["FastAPI", "pydantic"],
         "deliverables": ["API code", "OpenAPI spec", "curl examples"]},
    ],
    "flask": [
        {"title": "Mini Job Recommender Web App",
         "summary": "Deploy your Step-6 matcher as a Flask app.",
         "stack": ["Flask", "Python"],
         "deliverables": ["app.py", "procfile/uvicorn", "README"]},
    ],
    "powerpoint": [
        {"title": "Data Storytelling Deck",
         "summary": "10-slide narrative using a public dataset and clear visuals.",
         "stack": ["PowerPoint", "Excel/CSV"],
         "deliverables": [".pptx", "dataset link"]},
    ],
    # Backend / Web
    "node": [
        {"title": "RESTful CRUD Service",
         "summary": "JWT auth, CRUD, pagination, and error handling.",
         "stack": ["Node", "Express", "Postgres"],
         "deliverables": ["API", "tests", "README"]},
    ],
    "react": [
        {"title": "Job Search UI",
         "summary": "Search + filters + saved jobs; talks to your matcher API.",
         "stack": ["React", "Vite/Next"],
         "deliverables": ["SPA", "README"]},
    ],
    "typescript": [
        {"title": "Type-Safe Utils Library",
         "summary": "TS utility package with solid types & tests.",
         "stack": ["TypeScript", "Vitest"],
         "deliverables": ["NPM package", "docs site"]},
    ],
    # Non-technical tracks (examples)
    "marketing": [
        {"title": "Cohort-Based Email Funnel",
         "summary": "Plan & run a 4-email funnel; report open/click KPIs.",
         "stack": ["Canva", "Mailchimp"],
         "deliverables": ["email copy", "KPI report"]},
    ],
    "hr": [
        {"title": "Attrition Insights Brief",
         "summary": "Analyze sample HR data; produce a 2-page insights brief.",
         "stack": ["Excel", "PowerPoint"],
         "deliverables": ["xlsx", "pptx"]},
    ],
    "sales": [
        {"title": "Territory Planner",
         "summary": "Scoring model for leads + territory visualization.",
         "stack": ["Excel", "Power BI"],
         "deliverables": ["model", "dashboard"]},
    ],
}

def load_project_ideas(path: Path = PROJECT_IDEAS_PATH) -> Dict[str, List[Dict[str, Any]]]:
    """
    Load project ideas or create a default file if missing.
    The JSON schema is { skill: [ {title, summary, stack, deliverables}, ...], ... }
    """
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            ideas = json.load(f)
        # normalize keys
        return { _norm_skill(k): v for k, v in ideas.items() }
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(_DEFAULT_IDEAS, f, indent=2, ensure_ascii=False)
        return { _norm_skill(k): v for k, v in _DEFAULT_IDEAS.items() }

# ---------- Gap analysis ----------
def compute_missing_by_job(top_df: pd.DataFrame, resume_skills: List[str]) -> pd.DataFrame:
    """Add overlap/missing columns to top_df (non-destructive)."""
    have = set(_norm_skill(s) for s in _ensure_list(resume_skills))
    df = top_df.copy()
    skill_col = _choose_skill_col(df)
    def _row_sets(row):
        rskills = set(_norm_skill(s) for s in _ensure_list(row[skill_col]))
        overlap = sorted(rskills & have)
        missing = sorted(rskills - have)
        return pd.Series({
            "overlap_skills": overlap,
            "missing_skills": missing,
            "overlap_count": len(overlap),
            "missing_count": len(missing)
        })
    aug = df.apply(_row_sets, axis=1)
    return pd.concat([df.reset_index(drop=True), aug], axis=1)

def rank_missing_skills(df_aug: pd.DataFrame, k: int = 10) -> List[Tuple[str, float]]:
    """
    Score each missing skill by score-weighted frequency across top jobs.
    Falls back to unweighted frequency if no 'score' column is present.
    """
    has_score = "score" in df_aug.columns
    weights = []
    if has_score:
        s = df_aug["score"].astype(float)
        # normalize to 0..1 to avoid scale issues
        denom = s.max() if s.max() > 0 else 1.0
        weights = (s / denom).tolist()
    else:
        weights = [1.0] * len(df_aug)

    cw = defaultdict(float)
    for w, missing in zip(weights, df_aug["missing_skills"].tolist()):
        for m in _ensure_list(missing):
            cw[_norm_skill(m)] += float(w)

    ranked = sorted(cw.items(), key=lambda kv: kv[1], reverse=True)
    return ranked[:k]

# ---------- Recommendation selection ----------
def choose_projects_for_gaps(
    missing_ranked: List[Tuple[str, float]],
    ideas: Dict[str, List[Dict[str, Any]]],
    n_projects: int = N_PROJECTS
) -> List[Dict[str, Any]]:
    picks = []
    used_titles = set()

    for skill, _score in missing_ranked:
        if skill in ideas:
            random.shuffle(ideas[skill])
            for cand in ideas[skill]:
                title = cand["title"]
                if title not in used_titles:
                    picks.append({
                        "target_skill": skill,
                        **cand
                    })
                    used_titles.add(title)
                    break
        if len(picks) >= n_projects:
            break

    # If we couldn't fill enough because ideas are sparse, pad from any ideas
    if len(picks) < n_projects:
        pool = []
        for sk, lst in ideas.items():
            for cand in lst:
                pool.append({"target_skill": sk, **cand})
        random.shuffle(pool)
        for cand in pool:
            if cand["title"] not in used_titles:
                picks.append(cand)
            if len(picks) >= n_projects:
                break
    return picks[:n_projects]

def _depth_projects(have_skills: List[str], ideas: Dict[str, List[Dict[str, Any]]], n: int = 3):
    """When there are no gaps but portfolio is thin, propose depth projects in current stack."""
    have_norm = [_norm_skill(s) for s in have_skills]
    buckets = [s for s in have_norm if s in ideas]
    # pick a few skills the user already has
    random.shuffle(buckets)
    buckets = buckets[:max(1, min(3, len(buckets)))]
    picks = []
    used_titles = set()
    for sk in buckets:
        for cand in ideas.get(sk, []):
            if cand["title"] not in used_titles:
                picks.append({"target_skill": sk, **cand})
                used_titles.add(cand["title"])
                if len(picks) >= n:
                    break
        if len(picks) >= n:
            break
    return picks

# ---------- Orchestrator ----------
def recommend_projects(resume_text: str, field: str|None = None,
                       topn_jobs: int = TOPN_JOBS_FOR_GAPS,
                       n_projects: int = N_PROJECTS) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Main entry point for Step 7.
    Returns (projects_df, debug_payload)
    """
    # We already have these from Step 6:
    # - extract_resume_skills(resume_text) -> List[str]
    # - match_jobs(resume_text, field, topn) -> (top_jobs_df, have_skills)
    top_jobs, have_skills = match_jobs(resume_text, field=field, topn=topn_jobs)

    # Ensure overlap/missing are present
    df_aug = compute_missing_by_job(top_jobs, have_skills)

    # Rank the gaps
    missing_ranked = rank_missing_skills(df_aug, k=20)

    # Load ideas store (create default if missing)
    ideas = load_project_ideas(PROJECT_IDEAS_PATH)

    # Portfolio heuristic from your To-Do:
    # "If projects ≤ 2 AND missing ≈ ∅ → suggest depth projects; else 3–5 targeted ideas"
    est_proj_count = _estimate_project_count(resume_text)
    has_gaps = sum(cnt for _, cnt in missing_ranked) > 0

    if (est_proj_count <= 2) and (not has_gaps):
        picks = _depth_projects(have_skills, ideas, n=min(3, n_projects))
        mode = "depth"
    else:
        picks = choose_projects_for_gaps(missing_ranked, ideas, n_projects=n_projects)
        mode = "targeted"

    # Tidy output table
    projects_df = pd.DataFrame(picks)[
        ["title", "summary", "target_skill", "stack", "deliverables"]
    ]

    debug = {
        "mode": mode,
        "field": field,
        "estimated_project_count": est_proj_count,
        "resume_skills": sorted(set(_norm_skill(s) for s in have_skills)),
        "top_jobs_preview": df_aug.head(5).to_dict(orient="records"),
        "missing_ranked": missing_ranked[:10],
    }
    return projects_df, debug

# ---------- Example smoke test ----------
if __name__ == "__main__":
    demo = "Python, SQL, Tableau, Docker, AWS; built ETL pipelines and dashboards."
    projects, dbg = recommend_projects(demo, field="Data", topn_jobs=30, n_projects=5)
    print("\n=== Recommended Projects ===")
    print(projects.to_string(index=False))
    print("\n=== Debug (gap profile) ===")
    for k, v in dbg.items():
        print(f"{k}: {v if k != 'top_jobs_preview' else '[..rows..]'}")


AttributeError: 'DataFrame' object has no attribute 'tolist'